In [1]:
from pathlib import Path
import yaml
import uuid

In [2]:
FOLDER = Path("C:/____Moje-MOJE/MyProjects_4Fun/projects/World of Warcraft/rag-pliki/02_chunki")

In [3]:
records = list()

for path in FOLDER.rglob("*.md"):
    text = path.read_text(encoding="utf-8")
    parts = text.split("---", maxsplit=2)

    front_matter = parts[1].strip()
    body = parts[2].strip()

    metadata = yaml.safe_load(front_matter)
    record = {
        "id": metadata.get("chunk_id", "---NO ID"),
        "payload": metadata,
        "embedding_text": f"{metadata.get('chunk_title', '')}\n{body}"
    }

    records.append(record)

In [4]:
records

[{'id': 'chk_RadiantSong_001_overview',
  'payload': {'chunk_id': 'chk_RadiantSong_001_overview',
   'document_id': 'doc_RadiantSong_001',
   'source_type': 'chunk',
   'parent_source_url': 'https://warcraft.wiki.gg/wiki/Radiant_Song',
   'source_language': 'en',
   'entity_type': 'phenomenon',
   'entity_name': 'Radiant Song',
   'chunk_title': 'Radiant Song — overview and meaning',
   'chunk_role': 'phenomenon_overview',
   'topic': 'worldsoul_song_visions_warning'},
  'embedding_text': 'Radiant Song — overview and meaning\n# Radiant Song — overview and meaning\n\n## Contextual header\nRadiant Song, also interpretable in quest phrasing as the "song of the world", is a phenomenon connected to Azeroth\'s Worldsoul and experienced as a song or vision.\n\n## Chunk text\nThe Radiant Song, sometimes called Radiant Vision, is a phenomenon that manifests like a song or visions from someone calling out from the heart of the world. People across Azeroth report it as light-filled, benevolent, w

In [5]:
from fastembed import TextEmbedding

In [6]:
model = TextEmbedding(model_name="intfloat/multilingual-e5-large")

C:\Users\piotr\AppData\Local\Temp\ipykernel_29792\1956631489.py:1: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  model = TextEmbedding(model_name="intfloat/multilingual-e5-large")


In [7]:
passaged_list = [f"passage: {rec['embedding_text']}" for rec in records]
encoded = list(model.embed(passaged_list))

In [8]:
print(f"LEN RECORDS: {len(records)}")
#print(f"PASSAGED LIST: {len(passaged_list)}")
#print(f"LEN ENCODED: {len(encoded)}")
#print(f"LEN EMBEDDING: {len(encoded[0])}")

LEN RECORDS: 33


In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [10]:
client = QdrantClient(url="http://localhost:6333")

In [11]:
client.get_collections() # test polaczenia
# rezultat [] = Python połączył się z Qdrantem, ale nie ma jeszcze żadnych kolekcji.

CollectionsResponse(collections=[CollectionDescription(name='wow_lore_chunks')])

In [ ]:
COLLECTION_NAME = "wow_lore_chunks"

# client.create_collection(
#     collection_name=COLLECTION_NAME,
#     vectors_config=VectorParams(
#         size=1024,
#         distance=Distance.COSINE,
#     ),
# )

In [13]:
import uuid

In [14]:
points = []

if len(records) == len(encoded):
    for record, embedding in zip(records, encoded):
        chunk_id = record["payload"]["chunk_id"]
        point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id)) # stabilne id

        payload = record["payload"].copy()
        payload["embedding_config"] = {
            "embedding_model": "intfloat/multilingual-e5-large",
            "embedding_dim": 1024,
            "embedding_prefix": "passage"
        }
        payload["embedding_text"] = record["embedding_text"]

        point = PointStruct(        # tworzy jeden punkt qdranta, czyli jeden zapis w bazie wektorowej
            id=point_id,
            vector=list(embedding), # wektor chunka, czyli 1024 liczby z modelu E5
            payload=payload         # metadane i tekst chunka, np. chunk_title, entity_name, embedding_text
        )                           # to taki punkt w 1024 wymiarach; 1024 liczby wspólnie wyznaczają jeden punkt. Nie jako     przecięcie boków bryły, bardziej jako adres/współrzędne w bardzo wielowymiarowej mapie znaczeń.

        points.append(point)

client.upsert(# Wstaw punkt, a jeśli punkt o tym ID już istnieje, nadpisz go.
    collection_name=COLLECTION_NAME,
    points=points
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [15]:
client.count(collection_name=COLLECTION_NAME) # check czy dziala

CountResult(count=33)

In [16]:
query = "Why did Lightbloom begin behaving erratically near Fairbreeze Village after the Sunwell flared?"

In [17]:
embedded_query = list(model.embed([f"query: {query}"]))
query_vector = list(embedded_query[0])

In [18]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=10,
    with_payload=True,
)

In [19]:
points_unique = {}
for index, point in enumerate(results.points, start=1):
    payload = point.payload
    text_preview = payload["embedding_text"][:300].replace("\n", " ")

    points_unique[point.id] = {
        "score": f"{point.score:.3f}",
        "chunk_title": payload["chunk_title"],
        "entity_name": payload["entity_name"],
        "entity_type": payload["entity_type"],
        "topic": payload["topic"],
        "text_preview": text_preview,
    }

In [20]:
points_unique

{'6d37b5bb-9679-56b6-9941-76f8be5f1779': {'score': '0.905',
  'chunk_title': 'Lightbloom — Sunwell flare and Fairbreeze assault',
  'entity_name': 'Lightbloom',
  'entity_type': 'hostile_form',
  'topic': 'sunwell_flare_fairbreeze_hostile_expansion',
  'text_preview': 'Lightbloom — Sunwell flare and Fairbreeze assault # Lightbloom — Sunwell flare and Fairbreeze assault  ## Contextual header This chunk explains why Lightbloom becomes an active threat around Fairbreeze Village after the Sunwell flares during the Voidstorm.  ## Chunk text When the Sunwell erupted wit'},
 '007c4c79-c809-56b7-819a-16ca5ee67bae': {'score': '0.860',
  'chunk_title': 'Orweyna — Fairbreeze and the Lightbloom problem',
  'entity_name': 'Orweyna',
  'entity_type': 'character',
  'topic': 'midnight_fairbreeze_lightbloom_sunwell',
  'text_preview': 'Orweyna — Fairbreeze and the Lightbloom problem # Orweyna — Fairbreeze and the Lightbloom problem  ## Contextual header This chunk is directly relevant to Orweyna’s pre

In [21]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

In [ ]:
reranker = TextCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

In [23]:
documents = [
    point.payload["embedding_text"]
    for point in results.points
]

In [24]:
rerank_scores = list(reranker.rerank(query, documents))

In [25]:
reranked_results = []

for point, rerank_score in zip(results.points, rerank_scores):
    reranked_results.append({
        "qdrant_score": point.score,
        "rerank_score": rerank_score,
        "payload": point.payload,
    })

In [26]:
reranked_results = sorted(
    reranked_results,
    key=lambda item: item["rerank_score"],
    reverse=True,
)

In [27]:
chunks_to_prompt = []
previous_score = None

for index, item in enumerate(reranked_results, start=1):
    rerank_score = item["rerank_score"]
    payload = item["payload"]

    if index == 1:
        chunks_to_prompt.append(payload["embedding_text"])
        previous_score = rerank_score
        continue

    score_drop = previous_score - rerank_score

    if score_drop <= 0.35:
        chunks_to_prompt.append(payload["embedding_text"])
        previous_score = rerank_score
    else:
        break

In [28]:
chunks_to_prompt

['Orweyna — Fairbreeze and the Lightbloom problem\n# Orweyna — Fairbreeze and the Lightbloom problem\n\n## Contextual header\nThis chunk is directly relevant to Orweyna’s presence in Fairbreeze during Midnight and the Lightbloom problem around Eversong Woods.\n\n## Chunk text\nFollowing her departure from Undermine, Orweyna traveled far as she listened to the voice of her goddess and admitted that there was much to be learned from the surface lands. In time, her goddess led her to Fairbreeze Village as it dealt with a Lightbloom problem, and she reunited with the adventurer. She also met Arator, who revealed that Alleria had informed him of her.[44]\n\nMagistrix Landra Dawnstrider informed them that the Lightbloom had begun behaving erratically after the Sunwell flared up. Arator and Orweyna worked together to pacify the local animals disturbed by its presence.[45][46]\n\nThey tracked the displaced animals south and discovered a large overgrowth of Lightbloom, along with the bodies of 